In [2]:
import sys
!{sys.executable} -m pip install --user gradio

  Using cached pydub-0.25.1-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.6 kB)
  Using cached click-8.3.3-py3-none-any.whl.metadata (2.6 kB)
   ---------------------------------------- 0.0/19.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/19.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/19.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/19.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/19.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/19.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/19.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/19.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/19.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/19.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/19.7 MB ? eta -:--:--
    ---------------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.45.1 requires pandas<3,>=1.4.0, but you have pandas 3.0.2 which is incompatible.


In [1]:
import gradio as gr
import cv2
import numpy as np
import time
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from ultralytics import YOLO
import io

YOLO_MODEL_PATH  = r'C:\Users\Gaurav\OneDrive\Desktop\projects\Surface_anamoly_detection\best_yolov8n_steel_defect\weights\best.pt'
RTDETR_MODEL_PATH= r'C:\Users\Gaurav\OneDrive\Desktop\projects\Surface_anamoly_detection\rtdetr_l_steel_defect\weights\best.pt'

CLASS_NAMES = [
    'crescent_gap', 'dent', 'inclusion', 'oil_spot',
    'punching_hole', 'rolled_pit', 'silk_spot',
    'waist_folding', 'water_spot', 'welding_line'
]

YOLO_RESULTS = {
    'crescent_gap'  : 0.962,
    'dent'          : 0.974,
    'inclusion'     : 0.650,
    'oil_spot'      : 0.938,
    'punching_hole' : 0.951,
    'rolled_pit'    : 0.983,
    'silk_spot'     : 0.864,
    'waist_folding' : 0.971,
    'water_spot'    : 0.955,
    'welding_line'  : 0.892,
}
YOLO_OVERALL = 0.914

RTDETR_RESULTS = {
    'crescent_gap'  : 0.876,
    'dent'          : 0.850,
    'inclusion'     : 0.218,
    'oil_spot'      : 0.663,
    'punching_hole' : 0.924,
    'rolled_pit'    : 0.773,
    'silk_spot'     : 0.513,
    'waist_folding' : 0.795,
    'water_spot'    : 0.831,
    'welding_line'  : 0.877,
}
RTDETR_OVERALL = 0.732

CLASS_COLORS_BGR = [
    ( 56, 56,255),(51,157,255),(51,255,255),(51,200, 51),
    (255,255, 51),(255,153, 51),(255, 51,200),(200, 51,255),
    ( 50,140,255),(100,255,100),
]

print('Loading models...')
yolo_model   = YOLO(YOLO_MODEL_PATH)
rtdetr_model = YOLO(RTDETR_MODEL_PATH)
print('✅ Both models loaded')


# DETECTION FUNCTION
def run_detection(image, model, conf_threshold):
    """Run detection and return annotated image, report, time."""
    if image is None:
        return None, 'Upload an image', 0.0

    arr = np.array(image)
    bgr = cv2.cvtColor(arr, cv2.COLOR_RGB2BGR)

    t0      = time.time()
    results = model.predict(bgr, conf=conf_threshold,
                             iou=0.45, verbose=False)
    inf_ms  = (time.time() - t0) * 1000

    result    = results[0]
    annotated = bgr.copy()
    summary   = {}
    conf_list = []

    if result.boxes is not None and len(result.boxes):
        for box in result.boxes:
            cls_id       = int(box.cls.item())
            cf           = float(box.conf.item())
            x1,y1,x2,y2 = [int(v) for v in box.xyxy[0].tolist()]
            color        = CLASS_COLORS_BGR[cls_id % len(CLASS_COLORS_BGR)]

            cv2.rectangle(annotated,(x1,y1),(x2,y2),color,3)
            lbl = f'{CLASS_NAMES[cls_id]} {cf:.0%}'
            (tw,th),_ = cv2.getTextSize(lbl,
                cv2.FONT_HERSHEY_SIMPLEX, 0.65, 2)
            cv2.rectangle(annotated,(x1,y1-th-12),
                          (x1+tw+6,y1),color,-1)
            cv2.putText(annotated, lbl,(x1+3,y1-4),
                cv2.FONT_HERSHEY_SIMPLEX,0.65,(0,0,0),2)

            name = CLASS_NAMES[cls_id]
            summary[name] = summary.get(name,0)+1
            conf_list.append((cf, name))

    out_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)

    if summary:
        total  = sum(summary.values())
        report = f' {total} DEFECT(S) DETECTED\n'
        report += '─'*32+'\n'
        for k,v in sorted(summary.items()):
            report += f'  {k.upper():20s} ×{v}\n'
        report += '─'*32+'\n'
        report += 'Confidence Scores:\n'
        for cf,name in sorted(conf_list,reverse=True):
            report += f'  {cf:.1%}  {name}\n'
    else:
        report = f'NO DEFECTS DETECTED\n(above {conf_threshold:.0%} threshold)'

    return Image.fromarray(out_rgb), report, inf_ms


def detect_both(image, conf):
    """Run both models and return results."""
    yolo_img,   yolo_report,   yolo_ms   = run_detection(image, yolo_model,   conf)
    rtdetr_img, rtdetr_report, rtdetr_ms = run_detection(image, rtdetr_model, conf)

    yolo_info   = f' Inference: {yolo_ms:.0f}ms\nTest mAP@50: 91.4%\n\n{yolo_report}'
    rtdetr_info = f' Inference: {rtdetr_ms:.0f}ms\nTest mAP@50: 73.2%\n\n{rtdetr_report}'

    return yolo_img, yolo_info, rtdetr_img, rtdetr_info


def detect_single_yolo(image, conf):
    img, report, ms = run_detection(image, yolo_model, conf)
    return img, f' {ms:.0f}ms  |  mAP@50: 91.4%\n\n{report}'


def detect_single_rtdetr(image, conf):
    img, report, ms = run_detection(image, rtdetr_model, conf)
    return img, f' {ms:.0f}ms  |  mAP@50: 73.2%\n\n{report}'


# COMPARISON CHART GENERATOR
def make_comparison_chart():
    """Generate side-by-side comparison bar chart."""
    classes = CLASS_NAMES
    yolo_vals   = [YOLO_RESULTS[c]   for c in classes]
    rtdetr_vals = [RTDETR_RESULTS[c] for c in classes]

    x     = np.arange(len(classes))
    width = 0.38

    fig, ax = plt.subplots(figsize=(14, 6))
    fig.patch.set_facecolor('#0d1117')
    ax.set_facecolor('#0d1117')

    bars1 = ax.bar(x - width/2, yolo_vals,   width,
                   label='YOLOv8s',
                   color='#3b82f6', edgecolor='#1d4ed8',
                   linewidth=0.8)
    bars2 = ax.bar(x + width/2, rtdetr_vals, width,
                   label='RT-DETR',
                   color='#f59e0b', edgecolor='#b45309',
                   linewidth=0.8)

    ax.axhline(y=YOLO_OVERALL,   color='#3b82f6',
               linestyle='--', alpha=0.5, linewidth=1.5)
    ax.axhline(y=RTDETR_OVERALL, color='#f59e0b',
               linestyle='--', alpha=0.5, linewidth=1.5)

    ax.set_xticks(x)
    ax.set_xticklabels(
        [c.replace('_',' ').title() for c in classes],
        rotation=30, ha='right', fontsize=10, color='white')
    ax.set_ylabel('mAP@50', fontsize=12, color='white')
    ax.set_title('Per-Class mAP@50 — YOLOv8s vs RT-DETR',
                 fontsize=14, fontweight='bold', color='white', pad=15)
    ax.set_ylim(0, 1.12)
    ax.tick_params(colors='white')
    ax.spines['bottom'].set_color('#333')
    ax.spines['left'].set_color('#333')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.yaxis.set_tick_params(labelcolor='white')
    ax.grid(axis='y', alpha=0.15, color='white')

    for bar in bars1:
        h = bar.get_height()
        ax.text(bar.get_x()+bar.get_width()/2, h+0.01,
                f'{h:.0%}', ha='center', va='bottom',
                fontsize=7.5, color='#93c5fd', fontweight='bold')
    for bar in bars2:
        h = bar.get_height()
        ax.text(bar.get_x()+bar.get_width()/2, h+0.01,
                f'{h:.0%}', ha='center', va='bottom',
                fontsize=7.5, color='#fcd34d', fontweight='bold')

    legend = ax.legend(fontsize=12, framealpha=0.2,
                        labelcolor='white',
                        facecolor='#1a1a2e')

    ax.text(len(classes)-0.5, YOLO_OVERALL+0.02,
            f'YOLO avg {YOLO_OVERALL:.1%}',
            color='#93c5fd', fontsize=9)
    ax.text(len(classes)-0.5, RTDETR_OVERALL+0.02,
            f'RTDETR avg {RTDETR_OVERALL:.1%}',
            color='#fcd34d', fontsize=9)

    plt.tight_layout()
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=140,
                facecolor='#0d1117', bbox_inches='tight')
    buf.seek(0)
    plt.close()
    return Image.open(buf)


def make_overall_chart():
    """Overall mAP comparison bar."""
    fig, ax = plt.subplots(figsize=(7, 4))
    fig.patch.set_facecolor('#0d1117')
    ax.set_facecolor('#0d1117')

    models = ['YOLOv8s', 'RT-DETR']
    vals   = [YOLO_OVERALL, RTDETR_OVERALL]
    colors = ['#3b82f6', '#f59e0b']

    bars = ax.bar(models, vals, color=colors,
                  width=0.45, edgecolor='#ffffff22',
                  linewidth=0.8)
    ax.set_ylim(0, 1.15)
    ax.set_ylabel('mAP@50', color='white', fontsize=12)
    ax.set_title('Overall mAP@50 Comparison',
                 color='white', fontsize=13, fontweight='bold')
    ax.tick_params(colors='white')
    ax.spines['bottom'].set_color('#333')
    ax.spines['left'].set_color('#333')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.yaxis.set_tick_params(labelcolor='white')
    ax.xaxis.set_tick_params(labelcolor='white')
    ax.grid(axis='y', alpha=0.15, color='white')

    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2,
                val+0.02, f'{val:.1%}',
                ha='center', va='bottom',
                fontsize=16, fontweight='bold', color='white')

    diff = YOLO_OVERALL - RTDETR_OVERALL
    ax.annotate(f'YOLOv8s\n+{diff:.1%} better',
                xy=(0, YOLO_OVERALL), xytext=(0.5, 0.95),
                xycoords=('data','axes fraction'),
                textcoords='axes fraction',
                fontsize=10, color='#4ade80',
                ha='center',
                arrowprops=dict(arrowstyle='->',
                                color='#4ade80',
                                lw=1.5))

    plt.tight_layout()
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=140,
                facecolor='#0d1117', bbox_inches='tight')
    buf.seek(0)
    plt.close()
    return Image.open(buf)


def make_metrics_table():
    """Generate metrics comparison image."""
    fig, ax = plt.subplots(figsize=(10, 4))
    fig.patch.set_facecolor('#0d1117')
    ax.set_facecolor('#0d1117')
    ax.axis('off')

    columns = ['Metric', 'YOLOv8s', 'RT-DETR', 'Difference']
    rows = [
        ['mAP@50',        '91.4%', '73.2%', '+18.2% '],
        ['mAP@50:95',     '58.6%', '37.1%', '+21.5% '],
        ['Precision',     '90.3%', '74.2%', '+16.1% '],
        ['Recall',        '86.1%', '72.1%', '+14.0% '],
        ['Parameters',    '11.2M', '32M',   'YOLO smaller'],
        ['Architecture',  'CNN',   'Transformer', '-'],
        ['Training time', '~2 hrs','~3 hrs', 'YOLO faster'],
    ]

    table = ax.table(
        cellText=rows,
        colLabels=columns,
        cellLoc='center',
        loc='center',
    )
    table.auto_set_font_size(False)
    table.set_fontsize(11)
    table.scale(1, 2.0)

    for (row, col), cell in table.get_celld().items():
        cell.set_edgecolor('#334155')
        if row == 0:
            cell.set_facecolor('#1e3a5f')
            cell.set_text_props(color='white', fontweight='bold')
        elif col == 1:
            cell.set_facecolor('#0f2233')
            cell.set_text_props(color='#93c5fd')
        elif col == 2:
            cell.set_facecolor('#0f2233')
            cell.set_text_props(color='#fcd34d')
        elif col == 3:
            cell.set_facecolor('#0a1f0f')
            cell.set_text_props(color='#4ade80')
        else:
            cell.set_facecolor('#111827')
            cell.set_text_props(color='#e2e8f0')

    ax.set_title('Model Comparison Summary',
                 color='white', fontsize=13,
                 fontweight='bold', pad=15)

    plt.tight_layout()
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=140,
                facecolor='#0d1117', bbox_inches='tight')
    buf.seek(0)
    plt.close()
    return Image.open(buf)


# Pre-generate comparison charts
print('Generating comparison charts...')
comparison_chart = make_comparison_chart()
overall_chart    = make_overall_chart()
metrics_table    = make_metrics_table()
print('Charts ready')


# GRADIO UI
CSS = """
.gradio-container {
    background: #0a0e1a !important;
    font-family: 'Segoe UI', sans-serif;
}
.header-banner {
    background: linear-gradient(135deg, #0f2233 0%, #1a1a3e 50%, #0f2233 100%);
    border: 1px solid #1e3a5f;
    border-radius: 12px;
    padding: 24px 32px;
    margin-bottom: 16px;
}
.header-banner h1 {
    color: #e2e8f0;
    font-size: 1.8rem;
    margin: 0 0 6px 0;
}
.header-banner p {
    color: #94a3b8;
    margin: 0;
    font-size: 0.95rem;
}
.metric-card {
    background: #0f172a;
    border: 1px solid #1e3a5f;
    border-radius: 10px;
    padding: 16px 20px;
    text-align: center;
}
.metric-value {
    font-size: 2rem;
    font-weight: 800;
    display: block;
}
.metric-label {
    font-size: 0.8rem;
    color: #64748b;
    text-transform: uppercase;
    letter-spacing: 1px;
}
.yolo-accent { color: #3b82f6; }
.rtdetr-accent { color: #f59e0b; }
.win-accent { color: #4ade80; }
.tab-nav button {
    background: #1e293b !important;
    color: #94a3b8 !important;
    border: 1px solid #334155 !important;
}
.tab-nav button.selected {
    background: #1e3a5f !important;
    color: #e2e8f0 !important;
    border-color: #3b82f6 !important;
}
"""

with gr.Blocks(title='Steel Defect Inspector') as demo:

    #  Header 
    gr.HTML("""
    <div class="header-banner">
        <h1>🏭 Manufacturing Surface Anomaly Inspector</h1>
        <p>
            Deep Learning Object Detection &nbsp;|&nbsp;
            YOLOv8s vs RT-DETR &nbsp;|&nbsp;
            10 Defect Classes &nbsp;|&nbsp;
            GC10-DET Steel Dataset
        </p>
    </div>
    """)

    # ── Metric Cards ──────────────────────────────────────────────
    gr.HTML("""
    <div style="display:grid;grid-template-columns:repeat(4,1fr);gap:12px;margin-bottom:16px">
        <div class="metric-card">
            <span class="metric-value yolo-accent">91.4%</span>
            <span class="metric-label">YOLOv8s mAP@50</span>
        </div>
        <div class="metric-card">
            <span class="metric-value rtdetr-accent">73.2%</span>
            <span class="metric-label">RT-DETR mAP@50</span>
        </div>
        <div class="metric-card">
            <span class="metric-value win-accent">+18.2%</span>
            <span class="metric-label">YOLOv8s Advantage</span>
        </div>
        <div class="metric-card">
            <span class="metric-value" style="color:#a78bfa">10</span>
            <span class="metric-label">Defect Classes</span>
        </div>
    </div>
    """)

    with gr.Tabs(elem_classes='tab-nav'):

        # ── TAB 1: Side-by-Side Detection ─────────────────────────
        with gr.Tab('Live Detection — Both Models'):
            gr.Markdown("""
            ### Upload an image — both models run simultaneously
            Compare YOLOv8s and RT-DETR detections side by side
            """)
            with gr.Row():
                with gr.Column(scale=1):
                    inp_img  = gr.Image(type='pil',
                                        label='📷 Upload Steel Surface Image')
                    conf_sl  = gr.Slider(0.10, 0.90, value=0.25,
                                          step=0.05,
                                          label='Confidence Threshold')
                    btn_both = gr.Button('🔍 Detect with Both Models',
                                          variant='primary', size='lg')
                    gr.Markdown("""
                    **10 Detectable Classes:**
                    `crescent_gap` `dent` `inclusion` `oil_spot`
                    `punching_hole` `rolled_pit` `silk_spot`
                    `waist_folding` `water_spot` `welding_line`
                    """)

            with gr.Row():
                with gr.Column():
                    gr.HTML('<div style="text-align:center;color:#3b82f6;font-weight:700;font-size:1.1rem;padding:8px">🔵 YOLOv8s (Winner — 91.4% mAP)</div>')
                    yolo_out_img  = gr.Image(label='YOLOv8s Detection')
                    yolo_out_text = gr.Textbox(label='YOLOv8s Report',
                                               lines=12)
                with gr.Column():
                    gr.HTML('<div style="text-align:center;color:#f59e0b;font-weight:700;font-size:1.1rem;padding:8px">🟡 RT-DETR (73.2% mAP)</div>')
                    rtdetr_out_img  = gr.Image(label='RT-DETR Detection')
                    rtdetr_out_text = gr.Textbox(label='RT-DETR Report',
                                                  lines=12)

            btn_both.click(
                fn=detect_both,
                inputs=[inp_img, conf_sl],
                outputs=[yolo_out_img, yolo_out_text,
                          rtdetr_out_img, rtdetr_out_text]
            )

        #  TAB 2: YOLOv8s Only
        with gr.Tab('🔵 YOLOv8s Demo'):
            gr.Markdown('### YOLOv8s — Best Performer (91.4% mAP@50)')
            with gr.Row():
                with gr.Column(scale=1):
                    y_img  = gr.Image(type='pil',
                                       label='Upload Image')
                    y_conf = gr.Slider(0.10, 0.90, value=0.25,
                                        step=0.05,
                                        label='Confidence')
                    y_btn  = gr.Button('🔍 Detect',
                                        variant='primary', size='lg')
                with gr.Column(scale=1):
                    y_out_img  = gr.Image(label='Detection Result')
                    y_out_text = gr.Textbox(label='Detection Report',
                                             lines=15)
            y_btn.click(detect_single_yolo,
                         [y_img, y_conf],
                         [y_out_img, y_out_text])

        #  TAB 3: RT-DETR Only 
        with gr.Tab('🟡 RT-DETR Demo'):
            gr.Markdown('### RT-DETR — Transformer Architecture (73.2% mAP@50)')
            with gr.Row():
                with gr.Column(scale=1):
                    r_img  = gr.Image(type='pil',
                                       label='Upload Image')
                    r_conf = gr.Slider(0.10, 0.90, value=0.25,
                                        step=0.05,
                                        label='Confidence')
                    r_btn  = gr.Button('🔍 Detect',
                                        variant='primary', size='lg')
                with gr.Column(scale=1):
                    r_out_img  = gr.Image(label='Detection Result')
                    r_out_text = gr.Textbox(label='Detection Report',
                                             lines=15)
            r_btn.click(detect_single_rtdetr,
                         [r_img, r_conf],
                         [r_out_img, r_out_text])

        # ── TAB 4: Results Comparison ─────────────────────────────
        with gr.Tab('Results & Comparison'):
            gr.Markdown('### Algorithm Performance Comparison')
            with gr.Row():
                gr.Image(value=overall_chart, label='Overall mAP@50')
                gr.Image(value=metrics_table, label='Detailed Metrics')
                gr.Image(value=comparison_chart, label='Per-Class mAP@50 — YOLOv8s vs RT-DETR')
        #  TAB 5: About 
        with gr.Tab('ℹ️ About'):
            gr.Markdown("""
            ## Project Summary

            ### Problem
            Steel manufacturing produces defective surfaces. Manual inspection
            is slow and inconsistent. This system automates defect detection
            using deep learning object detection.

            ### Dataset — GC10-DET
            | Item | Detail |
            |------|--------|
            | Total Images | 2249 grayscale images |
            | After Augmentation | 4115+ balanced images |
            | Total Annotations | 6476 bounding boxes |
            | Classes | 10 defect types |
            | Format | Pascal VOC XML → YOLO TXT |

            ### Models Compared
            | Model | Architecture | mAP@50 | Winner |
            |-------|-------------|--------|--------|
            | **YOLOv8s** | CNN One-Stage | **91.4%** | |
            | RT-DETR | Transformer | 73.2% | |

            ### Key Techniques
            - **CLAHE** preprocessing for dark image enhancement
            - **Offline augmentation** to fix class imbalance (11 → 700 samples for Rolled Pit)
            - **Grayscale-safe training** — disabled hue/saturation augmentation
            - **AdamW optimizer** with cosine learning rate decay
            - **Early stopping** with patience=20 to prevent overfitting

            ### Best Result
            Rolled Pit improved from **0% mAP** (before balancing) to **98.3% mAP** 

            ### Tools Used
            Ultralytics YOLOv8 · PyTorch · Albumentations ·
            OpenCV · Gradio · Google Colab T4 GPU
            """)

    gr.HTML("""
    <div style="text-align:center;padding:16px;color:#475569;font-size:0.85rem;border-top:1px solid #1e293b;margin-top:16px">
        Manufacturing Surface Anomaly Inspector &nbsp;|&nbsp;
        YOLOv8s + RT-DETR &nbsp;|&nbsp;
        GC10-DET Dataset &nbsp;|&nbsp;
        Tesla T4 GPU Training
    </div>
    """)

print('Launching dashboard...')
demo.launch(share=True, debug=False, css=CSS)  #↑ PUBLIC URL appears here — share with teachers!

C:\ProgramData\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Loading models...
✅ Both models loaded
Generating comparison charts...
Charts ready
Launching dashboard...
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://73ef26022bb5c706f8.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
